# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We'll explore the data schema, load records, and perform basic analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}\n")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}\n")
print(f"License: {metadata.license}\n")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.


In [ ]:
# List available record sets and their fields

record_set_ids = []
if hasattr(metadata, 'recordSets'):
    record_sets = metadata.recordSets
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    record_sets = []

# If there are no attached recordSets, inspect the schema for minimal example
if not record_sets or len(record_sets) == 0:
    # Try to extract record set ids from serialization
    meta_dict = dataset.metadata.to_json()
    if 'recordSet' in meta_dict:
        record_sets = meta_dict['recordSet']
    elif 'recordSets' in meta_dict:
        record_sets = meta_dict['recordSets']
    else:
        print("No record sets available in this dataset.")
        record_sets = []

if not record_sets:
    print("No record sets found in this dataset. Dataset likely contains no tabular data.")
else:
    print("Record sets in the dataset:\n")
    for rs in record_sets:
        if isinstance(rs, dict):
            rs_id = rs.get('@id', str(rs))
        else:
            rs_id = str(rs)
        print(f"- {rs_id}")
        record_set_ids.append(rs_id)
        # Optionally display contained fields
        # fields = dataset.metadata.by_id(rs_id).fields if hasattr(dataset.metadata.by_id(rs_id), 'fields') else []

if not record_set_ids:
    # Try to guess a record set from content
    print("\nNo explicit recordSet declared. Attempting to infer record sets from distribution files.")
    meta_js = dataset.metadata.to_json()
    if 'distribution' in meta_js:
        for dist in meta_js['distribution']:
            file_id = dist.get('@id', None)
            print(f"Possible file for data: {file_id}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# In this dataset, there is no explicit recordSet declared in the Croissant metadata.
# However, we can try to infer the record set from the available distributions (typically data files).

meta_json = dataset.metadata.to_json()

# Extract distributions as possible sources of data
distribution_ids = []
if 'distribution' in meta_json:
    for dist in meta_json['distribution']:
        if '@id' in dist:
            distribution_ids.append(dist['@id'])

# If there are recordSets explicitly, use those
record_sets = meta_json.get('recordSet', []) if 'recordSet' in meta_json else []

# We prioritize recordSet ID if available, else use distribution file IDs
if record_sets:
    record_set_ids = [x['@id'] if isinstance(x, dict) and '@id' in x else x for x in record_sets]
else:
    record_set_ids = distribution_ids

print("Loading data from the following record or file IDs:")
for rec_id in record_set_ids:
    print(f"- {rec_id}")

# Try loading records from first record set or distribution
dataframes = {}
loaded = False
for record_set_id in record_set_ids:
    try:
        records = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(list(records))
        dataframes[record_set_id] = df
        if not df.empty:
            loaded = True
            print(f"Successfully loaded {len(df)} records from {record_set_id}")
            print(f"Sample columns: {df.columns.tolist()}")
        else:
            print(f"Loaded 0 records from {record_set_id}")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

# Choose first loaded dataframe for further analysis
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nMain table record set or distribution ID: {main_record_set_id}")
    print(df.head())
else:
    print("No suitable data frame loaded for analysis.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Find a numeric field in the loaded dataframe for filtering & normalization
import numpy as np

if main_record_set_id is None:
    print("No data available for EDA.")
else:
    df = dataframes[main_record_set_id]
    # Attempt to identify a numeric field to process
    example_numeric = None
    for col in df.columns:
        # Heuristics: look for common numeric-type field names
        if df[col].dtype in [np.float64, np.int64, float, int]:
            example_numeric = col
            break
        if any(substr in col.lower() for substr in ["count", "coverage", "mw", "weight", "abundance"]):
            # Try to coerce to numeric
            try:
                _ = pd.to_numeric(df[col].dropna().iloc[:5])
                example_numeric = col
                break
            except Exception:
                pass
    if example_numeric is None:
        print("No numeric column was found for analysis.\nColumns: ", df.columns.tolist())
    else:
        # If possible, convert this column to numeric type
        df[example_numeric] = pd.to_numeric(df[example_numeric], errors='coerce')
        # Filter: values > a threshold (e.g., 10)
        threshold = 10
        filtered_df = df[df[example_numeric] > threshold]
        print(f"Filtered records with {example_numeric} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{example_numeric}_normalized"] = (filtered_df[example_numeric] - filtered_df[example_numeric].mean()) / filtered_df[example_numeric].std()
        print(f"\nNormalized {example_numeric} for filtered records:")
        print(filtered_df[[example_numeric, f"{example_numeric}_normalized"]].head())

        # Try grouping by another field (e.g., 'description', 'accession')
        group_field = None
        for col in ['description', 'Description', 'gene', 'accession', 'sample', 'Sample']:
            if col in df.columns:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (means):")
            print(grouped_df.head())
        else:
            print("No suitable group field present for grouping.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is None or example_numeric is None:
    print("Visualization skipped: no suitable numeric data loaded.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[example_numeric].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {example_numeric}")
    plt.xlabel(example_numeric)
    plt.ylabel("Count")
    plt.show()

    # If grouped, show mean per group (top 10)
    if group_field is not None:
        group_means = df.groupby(group_field)[example_numeric].mean(numeric_only=True).sort_values(ascending=False)[:10]
        group_means.plot(kind='bar', figsize=(8,3), title=f"Top 10 {group_field} by average {example_numeric}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {example_numeric}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
We demonstrated how to use the `mlcroissant` library to explore this mass spectrometry dataset. With the schema and data loaded, we reviewed potential record sets, analyzed key numeric fields, and visualized distributions. This workflow can be adapted for deeper analysis, integration, or modeling on structured FAIR-compliant datasets.
